# 04 - Modelling, explainability & business impact (retail churn)

Run `python -m src.pipeline` first - this notebook reads what it produced (`reports/model_comparison.json`, `reports/threshold_economics.json`, scored customers, figures).

**Protocol:** stratified 80/20 split -> randomised search per model family (`SEARCH_ITERATIONS` draws, `CV_FOLDS`-fold stratified CV on the training split, optimising `SELECTION_METRIC`) -> money-optimal threshold chosen on out-of-fold training predictions -> the test split is scored once, at the end.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import Image, display
try:
    from src import configs
except ImportError:
    from src import configs as config   # same file, saved under either name

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

T = config.TARGET

def show(name):
    p = config.FIGURES_DIR / name
    if p.exists():
        display(Image(str(p)))
    else:
        print(f"missing figure: {name} - run the pipeline first")

report = json.loads(config.METRICS_JSON.read_text())
econ = json.loads(config.THRESHOLD_JSON.read_text())
print("champion:", report["champion"])
print(f"features: {len(config.NUMERIC_FEATURES)} numeric ({len(config.ENGINEERED_NUMERIC)} engineered) + {len(config.CATEGORICAL_FEATURES)} categorical")

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\customer_churn\\reports\\model_comparison.json'

## Model comparison

In [ ]:
rows = []
for name, m in report["models"].items():
    d = m["test_at_0.5"]
    rows.append({"model": name,
                 "cv_roc_auc": m["cv"]["roc_auc_mean"], "cv_std": m["cv"]["roc_auc_std"],
                 "test_roc_auc": d["roc_auc"], "test_pr_auc": d["pr_auc"],
                 "accuracy": d["accuracy"], "precision": d["precision"], "recall": d["recall"],
                 "f1": d["f1"], "brier": d["brier"],
                 "t*": m["optimal_threshold"],
                 "net_value_at_t*": m["campaign_test_at_optimal"]["net_value"]})
cmp = pd.DataFrame(rows).set_index("model")
display(cmp.round(3))

ranked = cmp.sort_values("cv_roc_auc", ascending=False)
k = min(3, len(ranked) - 1)
print(f"CV ROC-AUC: best {ranked['cv_roc_auc'].iloc[0]:.4f}, gap to #{k + 1}: "
      f"{ranked['cv_roc_auc'].iloc[0] - ranked['cv_roc_auc'].iloc[k]:.4f}, "
      f"typical CV std: {ranked['cv_std'].mean():.4f}")

In [ ]:
show("model_roc_pr.png")

*Fill in after running:*

1. **Champion vs the rest:** is the champion's CV ROC-AUC lead larger than the CV std? If the top models are within noise of each other, say so - a simpler model (logistic regression) is then a defensible choice.
2. **Test vs CV:** does test ROC-AUC agree with CV? A big gap means the split or the tuning overfit.
3. **Net value by model:** differences of a few percent at this sample size are noise; the champion is chosen on CV, not on test.

## Calibration - can the probabilities be used as money?

In [ ]:
show("model_calibration.png")
pd.DataFrame(report["imbalance_experiment"]).round(4)

*Fill in after running:* is the unweighted model's average predicted probability close to the true churn rate (`feats[T].mean()`)? Compare ROC-AUC, mean prediction and Brier score with the class-weighted variants.

Expected loss = probability x customer value, so the project keeps the unweighted, calibrated model and handles imbalance at the **decision threshold**. Re-weighting classes changes the probability scale, which would corrupt the money numbers.

## Choosing the threshold with money, not accuracy

The economics are in `config.ECONOMICS`: one-off `offer_cost` per contacted customer, a `save_rate` for contacted churners, and customer value = spend per year x (`horizon_months` / 12) x `gross_margin`.

In [ ]:
print(config.ECONOMICS.as_dict())
print(json.dumps(econ["economics"], indent=2))
print("champion t*:", report["models"][report["champion"]]["optimal_threshold"])
show("threshold_economics.png")
pd.DataFrame(econ["policy_comparison_test"])[["policy", "contacted", "churners_caught",
    "precision", "recall", "retained_value", "campaign_cost", "net_value"]].round(3)

*Fill in after running:*

- **Threshold:** the value chosen on training out-of-fold predictions, and whether the OOF and test net-value curves overlap (i.e. whether the choice generalises).
- **vs 0.5:** churners reached and net value at t* vs at 0.5, in dollars and percent.
- **Plateau:** how wide is the near-optimal range? A broad plateau means "never default to 0.5 when a miss costs more than a false alarm" matters more than the exact number.
- **Expected-value rule** (contact when P x save_rate x value > offer_cost): how does it compare with the single threshold? It spends the offer budget on valuable customers, and only works because the probabilities are calibrated.

## Explainability - global

In [ ]:
show("shap_global_importance.png")
show("shap_beeswarm.png")
pd.read_csv(config.REPORTS_DIR / "permutation_importance.csv").head(10).round(4)

*Fill in after running:* top drivers by SHAP and by permutation importance, and whether they agree with the EDA (recency, satisfaction, support contacts, returns). Several features are correlated (e.g. `total_spend`, `spend_per_year`, `num_of_purchases`, `purchases_per_year`), so importance is shared between them - read it per business theme, not per column.

## Explainability - one customer, and the example prediction output

In [ ]:
show("shap_customer_example.png")
scored = pd.read_csv(config.SCORED_CSV)
active = scored[scored[T] == 0].sort_values("expected_loss", ascending=False)
c = active.iloc[0]
months = config.ECONOMICS.horizon_months
print(f"""Customer ID: {c[config.ID_COL]}

Churn Probability: {c.churn_probability:.1%}
Risk Level: {c.risk_level}
Expected margin loss ({months} months): ${c.expected_loss:,.0f}

Major Risk Factors:
1. {c.reason_1}
2. {c.reason_2}
3. {c.reason_3}

Recommended Action:
{c.recommended_action.replace(' | ', chr(10))}""")

## Who to call first - ranked by expected loss

In [ ]:
wanted = [config.ID_COL, "churn_probability", "risk_level", "expected_loss", "years_as_customer",
          "spend_per_year", "satisfaction_score", "last_purchase_days_ago", "top_reasons"]
cols = [c for c in wanted if c in active.columns]
active[active["contact_recommended"]][cols].head(15).round(3)

In [ ]:
print(f"active customers: {len(active):,}")
print(f"recommended for contact: {active['contact_recommended'].sum():,}")
print(f"expected margin loss, all active: ${active['expected_loss'].sum():,.0f}")
print(f"  ...of which in the contact list: ${active.loc[active['contact_recommended'], 'expected_loss'].sum():,.0f}")
print()
print("Risk bands (cutoffs in config.RISK_BANDS):")
active.groupby("risk_level").agg(customers=(config.ID_COL, "size"),
                                 expected_loss=("expected_loss", "sum")).round(0)

## Segments

In [ ]:
show("segments.png")
pd.read_csv(config.REPORTS_DIR / "segment_profile.csv").round(3)

*Fill in after running:* which segment holds the most expected loss, and how it differs from the others (recency, satisfaction, spend). Name the action each segment should get.

## From SHAP reasons to a retention message (LLM)

In [ ]:
print("model:", config.LLM_MODEL)
for ex in json.loads(config.LLM_EXAMPLES_JSON.read_text())[:2]:
    print(ex["context"]["customer_id"], "| source:", ex["brief"]["source"])
    print("SUMMARY:", ex["brief"]["summary"])
    print("MESSAGE:", ex["brief"]["customer_message"], end="\n\n")

The LLM never sees raw data - only the model's probability, the SHAP reasons and the approved playbook actions - and is told not to invent offers. Without an `ANTHROPIC_API_KEY` the same fields come from a deterministic template.